In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import warnings
warnings.filterwarnings("ignore")

%cd /content/drive/MyDrive/VedasFineTune

In [ ]:
!pip install --upgrade "transformers>=4.46.0" "trl>=0.15.0"
!pip install --upgrade unsloth unsloth_zoo
!pip install --upgrade --no-deps "unsloth[cu121-torch240] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-deps "xformers<0.0.28" peft accelerate bitsandbytes

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
import torch

### 1. Load the Instruct Model

In [ ]:
max_seq_length = 2048
dtype = None  # Auto-detect. Float16 for T4, BFloat16 for Ampere+
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 2. Configure LoRA

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 3. Apply Chat Template

In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3",
)
print("✅ Chat template applied to tokenizer.")

### 4. Overlapping Chunking Function

In [ ]:
def chunk_with_overlap(examples, chunk_size=512, overlap_pct=0.2):
    """Chunk raw text with overlap. Run BEFORE formatting."""
    overlap_size = int(chunk_size * overlap_pct)
    chunked_texts = []
    chunked_collections = []
    chunked_books = []

    for i in range(len(examples['text'])):
        text = examples['text'][i]
        collection = examples.get('collection', ['Unknown'])[i]
        book = examples.get('book', [''])[i]

        # ~4 chars per token heuristic
        stride = (chunk_size - overlap_size) * 4
        limit = chunk_size * 4

        start = 0
        while start < len(text):
            end = start + limit
            chunk = text[start:end]
            chunked_texts.append(chunk)
            chunked_collections.append(collection)
            chunked_books.append(book)
            if end >= len(text):
                break
            start += stride

    return {
        "text": chunked_texts,
        "collection": chunked_collections,
        "book": chunked_books,
    }

### 5. Chat Formatting Function

In [ ]:
def formatting_pretrain_function(example):
    """Wrap book text in chat format so the model learns content
    WITHOUT forgetting how to chat."""
    collection = example.get('collection', 'Unknown')
    book = example.get('book', '')

    messages = [
        {
            "role": "system",
            "content": "You are VedaGPT, an expert scholar of the ancient Vedic scriptures. "
                       "Share the sacred verses faithfully and accurately."
        },
        {
            "role": "user",
            "content": f"Recite a passage from {collection}, Book {book}."
        },
        {
            "role": "assistant",
            "content": example["text"]
        },
    ]

    full_text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": full_text}

### 6. Load, Chunk, and Format Dataset

In [ ]:
# 1. Load raw data
print("Loading dataset...")
dataset_train = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/VedasFineTune/continueousPreTrainData.jsonl",
    split="train",
)
print(f"Raw examples: {len(dataset_train)}")

# 2. Chunk raw text (with overlap)
print("Chunking with 20% overlap...")
dataset_train = dataset_train.map(
    chunk_with_overlap,
    batched=True,
    remove_columns=dataset_train.column_names,
)
print(f"After chunking: {len(dataset_train)}")

# 3. Format into chat template (done ONCE)
print("Formatting into chat template...")
dataset_train = dataset_train.map(formatting_pretrain_function, batched=False)
print(f"Final training examples: {len(dataset_train)}")

# 4. Verify
print("\n--- WHAT THE MODEL SEES DURING TRAINING ---")
print(dataset_train[0]['text'][:500])

### 7. Setup SFTTrainer

In [ ]:
# Detect BF16 support (False on Tesla T4)
use_bf16 = torch.cuda.is_bf16_supported()

for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset_train,
    processing_class = tokenizer,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = max_seq_length,
        dataset_num_proc = 2,
        packing = False,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not use_bf16,
        bf16 = use_bf16,
        logging_steps = 5,
        output_dir = "outputs",
        optim = "adamw_8bit",
        seed = 3407,
        report_to = "none",
    ),
)

### 8. Train

In [ ]:
print("Starting training...")
model.for_training()
trainer_stats = trainer.train()

### 9. Save Model and Tokenizer

In [ ]:
# Save LoRA adapters + tokenizer locally
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("✅ Model and tokenizer saved to lora_model/")

# Also save to Google Drive for persistence
import shutil, os
drive_path = "/content/drive/MyDrive/VedasFineTune/lora_model"
os.makedirs(drive_path, exist_ok=True)

# Copy all files
for f in os.listdir("lora_model"):
    shutil.copy2(os.path.join("lora_model", f), drive_path)
print(f"✅ Backed up to {drive_path}")

### 10. Inference Test

In [ ]:
FastLanguageModel.for_inference(model)

def ask_vedagpt(query):
    """Chat with VedaGPT using proper chat template."""
    messages = [
        {
            "role": "system",
            "content": "You are VedaGPT, an expert scholar of the ancient Vedic scriptures. "
                       "Answer questions accurately based on the Vedas."
        },
        {"role": "user", "content": query},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 256,
        use_cache = True,
        temperature = 0.7,
        do_sample = True,
        top_p = 0.9,
    )

    # Decode only the generated tokens (skip the prompt)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    return response.strip()

# Test queries
for q in [
    "What are the four Vedas?",
    "Recite Hymn 1 from the Rig Veda.",
    "Which vedas do you know?",
]:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"A: {ask_vedagpt(q)}")

### 11. Verify Chat Template

In [ ]:
import pprint

print("--- CHAT TEMPLATE ---")
pprint.pprint(tokenizer.chat_template)

# Test formatting
messages = [
    {"role": "user", "content": "What are the four Vedas?"},
]
formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print("\n--- FORMATTED PROMPT ---")
print(formatted)

### 12. Deploy to HuggingFace

In [ ]:
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN environment variable not set. Please set it in your environment or .env file.")

# Find the saved model
possible_paths = [
    "/content/drive/MyDrive/VedasFineTune/lora_model",
    "/content/lora_model",
    "lora_model",
]
model_path = None
for path in possible_paths:
    if os.path.exists(os.path.join(path, "adapter_config.json")):
        model_path = path
        break

if model_path is None:
    raise FileNotFoundError("Could not find lora_model with adapter_config.json")

print(f"Loading from: {model_path}")

# Reload
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

# Re-apply chat template (in case tokenizer_config.json didn't persist it)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

# 1. Push LoRA adapters
print("Pushing LoRA adapters...")
model.push_to_hub("shinigamiRaj/Vedas-Llama-3.2-3B-LoRA", token=hf_token)
tokenizer.push_to_hub("shinigamiRaj/Vedas-Llama-3.2-3B-LoRA", token=hf_token)

# 2. Push merged 16-bit model
print("Pushing merged model...")
model.push_to_hub_merged(
    "shinigamiRaj/Vedas-Llama-3.2-3B-Merged",
    tokenizer,
    save_method = "merged_16bit",
    token = hf_token,
)

# 3. Export GGUF for LM Studio / Ollama
print("Exporting GGUF...")
model.save_pretrained_gguf(
    "model_gguf",
    tokenizer,
    quantization_method = "q4_k_m",
)

print("✅ All formats exported!")